# Traffic Violation Detection & Fine Generation System

**Architecture:** YOLOv8n → ByteTrack → Violation Engine → PaddleOCR → Fine Generation


In [ ]:
!pip install -q paddlepaddle-gpu==3.1.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ 2>/dev/null ||  pip install -q paddlepaddle-gpu==3.1.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ 2>/dev/null ||  pip install -q paddlepaddle==3.1.1
!pip install -q ultralytics
!pip install -q 'paddleocr==3.7.0'
!pip install -q roboflow
!pip install -q lapx

import paddle, paddleocr
print(f"paddle    : {paddle.__version__}")    # must be >= 3.1.0
print(f"paddleocr : {paddleocr.__version__}") # must be >= 3.0.0

_major, _minor = (int(x) for x in paddle.__version__.split('.')[:2])
assert (_major, _minor) >= (3, 1), (
    f"paddlepaddle >= 3.1.0 required (paddleocr 3.7.0 only fully supports paddle 3.1.x), got {paddle.__version__}.\n"
    "Run: !pip install paddlepaddle-gpu==3.1.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/  then restart the runtime."
)

import paddle.inference as _pi
_cfg = _pi.Config()
assert hasattr(_cfg, 'set_optimization_level'), "set_optimization_level missing — paddle version too old"
print("All version checks passed ✓")

## Imports & Global Config

In [ ]:
import os
import cv2
import yaml
import shutil
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from ultralytics import YOLO
from IPython.display import Video, display

import warnings
warnings.filterwarnings('ignore')

sns.set(rc={'axes.facecolor': '#eae8fa'}, style='darkgrid')
print('All imports OK')

## 1. Load Pre-trained Backbone

In [ ]:
model = YOLO('yolov8n.pt')
print(f'Model loaded: {model.info()}')

## 2. Dataset Download via Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="TdOKixM39Uh1tATTadka")
project = rf.workspace("mds-workspace-pgksy").project("traffic-ahyux-lyqnf")
version = project.version(2)
dataset = version.download("yolov8")

dataset_path = dataset.location
print(f'Dataset downloaded to: {dataset_path}')

## 3. Pre-training Sanity Check

In [ ]:
train_dir = os.path.join(dataset_path, 'train', 'images')

train_images = [
    f for f in os.listdir(train_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

if train_images:
    img_path = os.path.join(train_dir, train_images[0])
    results = model.predict(source=img_path, imgsz=640, conf=0.5, verbose=False)
    simple_image = results[0].plot(line_width=2)
    simple_image = cv2.cvtColor(simple_image, cv2.COLOR_BGR2RGB)
    print(f'Predicted on: {img_path}')
else:
    print(f'No images found in: {train_dir}')
    simple_image = None

In [ ]:
if simple_image is not None:
    plt.figure(figsize=(20, 15))
    plt.imshow(simple_image)
    plt.title('Pretrained YOLOv8n Prediction on COCO Weights (before fine-tuning)', fontsize=16)
    plt.axis('off')
    plt.show()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
train_images_path = os.path.join(dataset_path, 'train', 'images')
val_images_path   = os.path.join(dataset_path, 'valid', 'images')
test_images_path  = os.path.join(dataset_path, 'test',  'images')

def count_and_sizes(folder):
    sizes = set()
    count = 0
    if not os.path.exists(folder):
        return 0, sizes
    for f in os.listdir(folder):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            count += 1
            try:
                with Image.open(os.path.join(folder, f)) as img:
                    sizes.add(img.size)
            except Exception:
                continue
    return count, sizes

n_train, train_sizes = count_and_sizes(train_images_path)
n_val,   val_sizes   = count_and_sizes(val_images_path)
n_test,  test_sizes  = count_and_sizes(test_images_path)

print(f'Train images : {n_train}')
print(f'Val   images : {n_val}')
print(f'Test  images : {n_test}')
print(f'Train sizes  : {train_sizes}')
print(f'Val   sizes  : {val_sizes}')


In [ ]:
image_files = [
    f for f in os.listdir(train_images_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]
num_images = len(image_files)
step = max(1, num_images // 8)
selected_images = image_files[::step][:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 11))
for i, ax in enumerate(axes.ravel()):
    if i < len(selected_images):
        img_path = os.path.join(train_images_path, selected_images[i])
        ax.imshow(Image.open(img_path))
        ax.set_title(f'Sample {i+1}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Images from Training Dataset', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
train_labels_path = os.path.join(dataset_path, 'train', 'labels')
class_counts = {}

yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg.get('names', [])

if os.path.exists(train_labels_path):
    for label_file in os.listdir(train_labels_path):
        if not label_file.endswith('.txt'):
            continue
        with open(os.path.join(train_labels_path, label_file)) as f:
            for line in f:
                cls_id = int(line.strip().split()[0])
                class_counts[cls_id] = class_counts.get(cls_id, 0) + 1

    labels = [class_names[k] if k < len(class_names) else str(k) for k in sorted(class_counts)]
    counts = [class_counts[k] for k in sorted(class_counts)]

    plt.figure(figsize=(12, 5))
    sns.barplot(x=labels, y=counts, palette='viridis')
    plt.title('Class Distribution in Training Set', fontsize=16)
    plt.xlabel('Class')
    plt.ylabel('Instance Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print('Classes found:', dict(zip(labels, counts)))
else:
    print('Label folder not found — skipping class distribution plot')

## 5. Fix data.yaml Paths

In [ ]:
yaml_path = os.path.join(dataset_path, 'data.yaml')

with open(yaml_path, 'r') as f:
    content = yaml.safe_load(f)

content['train'] = os.path.join(dataset_path, 'train', 'images')
content['val']   = os.path.join(dataset_path, 'valid', 'images')
content['test']  = os.path.join(dataset_path, 'test',  'images')

with open(yaml_path, 'w') as f:
    yaml.dump(content, f, default_flow_style=False)

print('data.yaml updated:')
with open(yaml_path) as f:
    print(f.read())

## 6. Fine-tune YOLOv8n

In [ ]:
train_results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    device=0,
    batch=16,
    patience=30,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10.0,
    dropout=0.1,
    seed=42,
    val=True,
    fraction=1.0,
    project='runs/detect',
    name='traffic_violation',
    exist_ok=True,
)

post_training_files_path = str(train_results.save_dir)
print(f'Training artifacts saved to: {post_training_files_path}')

## 7. Post-Training Analysis

In [ ]:
def plot_learning_curve(df, train_col, val_col, title):
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=df, x='epoch', y=train_col,
                 label='Train Loss', color='#141140', linestyle='-', linewidth=2)
    sns.lineplot(data=df, x='epoch', y=val_col,
                 label='Validation Loss', color='orangered', linestyle='--', linewidth=2)
    plt.title(title, fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

results_csv_path = os.path.join(post_training_files_path, 'results.csv')
df = pd.read_csv(results_csv_path)
df.columns = df.columns.str.strip()
print('Columns available:', df.columns.tolist())

plot_learning_curve(df, 'train/box_loss', 'val/box_loss', 'Box Loss Learning Curve')
plot_learning_curve(df, 'train/cls_loss', 'val/cls_loss', 'Classification Loss Learning Curve')
plot_learning_curve(df, 'train/dfl_loss', 'val/dfl_loss', 'DFL Loss Learning Curve')

In [ ]:
def show_training_plot(filename, title):
    path = os.path.join(post_training_files_path, filename)
    if os.path.exists(path):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14)
        plt.show()
    else:
        print(f'Plot not found: {path}')

show_training_plot('confusion_matrix_normalized.png', 'Normalised Confusion Matrix')
show_training_plot('PR_curve.png', 'Precision–Recall Curve')
show_training_plot('F1_curve.png', 'F1 Curve')
show_training_plot('results.png', 'Training Results Summary')


## 8. Load Best Weights & Validate

In [ ]:
best_model_path = os.path.join(post_training_files_path, 'weights', 'best.pt')
print(f'Loading best model from: {best_model_path}')

best_model = YOLO(best_model_path)
metrics = best_model.val(data=yaml_path, split='val', imgsz=640, conf=0.5)
metrics_df = pd.DataFrame.from_dict(
    metrics.results_dict, orient='index', columns=['Metric Value']
).round(4)
print(metrics_df.to_string())
metrics_df

## 9. Inference on Validation Images

In [ ]:
val_image_files = [
    f for f in os.listdir(val_images_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]
num_val = len(val_image_files)
selected_val = val_image_files[::max(1, num_val // 9)][:9]

fig, axes = plt.subplots(3, 3, figsize=(20, 21))
fig.suptitle('Validation Set Inferences (best.pt)', fontsize=22)

for i, ax in enumerate(axes.flatten()):
    if i < len(selected_val):
        image_path = os.path.join(val_images_path, selected_val[i])
        res = best_model.predict(source=image_path, imgsz=640, conf=0.5, verbose=False)
        annotated = cv2.cvtColor(res[0].plot(line_width=1), cv2.COLOR_BGR2RGB)
        ax.imshow(annotated)
        ax.set_title(selected_val[i][:30], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 10. Inference on a Test Image

In [ ]:
test_image_files = [
    f for f in os.listdir(test_images_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

if test_image_files:
    sample_image_path = os.path.join(test_images_path, test_image_files[0])
    res = best_model.predict(source=sample_image_path, imgsz=640, conf=0.5, verbose=False)
    sample_image_rgb = cv2.cvtColor(res[0].plot(line_width=2), cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(20, 15))
    plt.imshow(sample_image_rgb)
    plt.title('Test Image — Fine-tuned YOLOv8n', fontsize=18)
    plt.axis('off')
    plt.show()
else:
    print('No test images found.')


## 11. Video Inference with Lane-Wise Traffic Density

In [ ]:
heavy_traffic_threshold = 10

vertices1 = np.array([(465, 350), (609, 350), (510, 630), (2,   630)], dtype=np.int32)
vertices2 = np.array([(678, 350), (815, 350), (1203, 630), (743, 630)], dtype=np.int32)

roi_top, roi_bottom = 325, 635
lane_threshold = 609

font            = cv2.FONT_HERSHEY_SIMPLEX
font_scale      = 1
font_color      = (255, 255, 255)
background_color = (0, 0, 200)

video_path = '/content/Cars.mp4'

if not os.path.exists(video_path):
    print(f'Video not found at {video_path}. Upload it to /content/ first.')
else:
    cap = cv2.VideoCapture(video_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 20.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter('traffic_density_analysis_raw.mp4', fourcc, fps, (width, height))

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        detection_frame = frame.copy()
        detection_frame[:roi_top,  :] = 0
        detection_frame[roi_bottom:, :] = 0

        res = best_model.predict(detection_frame, imgsz=640, conf=0.4, verbose=False)
        processed_frame = res[0].plot(line_width=1)

        processed_frame[:roi_top,  :] = frame[:roi_top,  :].copy()
        processed_frame[roi_bottom:, :] = frame[roi_bottom:, :].copy()

        cv2.polylines(processed_frame, [vertices1], True, (0, 255, 0), 2)
        cv2.polylines(processed_frame, [vertices2], True, (255, 0, 0), 2)

        boxes = res[0].boxes
        left_count = right_count = 0
        if boxes is not None and boxes.xyxy is not None:
            for box in boxes.xyxy.cpu().numpy():
                cx = (box[0] + box[2]) / 2
                if cx < lane_threshold:
                    left_count += 1
                else:
                    right_count += 1

        intensity_left  = 'Heavy' if left_count  > heavy_traffic_threshold else 'Smooth'
        intensity_right = 'Heavy' if right_count > heavy_traffic_threshold else 'Smooth'

        def draw_hud(frame, text, pos):
            x, y = pos
            (w, h), _ = cv2.getTextSize(text, font, font_scale, 2)
            cv2.rectangle(frame, (x - 5, y - h - 5), (x + w + 5, y + 5), background_color, -1)
            cv2.putText(frame, text, (x, y), font, font_scale, font_color, 2, cv2.LINE_AA)

        draw_hud(processed_frame, f'Left  Lane: {left_count}  [{intensity_left}]',  (10,  45))
        draw_hud(processed_frame, f'Right Lane: {right_count} [{intensity_right}]', (10,  90))

        out.write(processed_frame)

    cap.release()
    out.release()
    print(f'Processed {frame_count} frames → traffic_density_analysis_raw.mp4')

In [ ]:
!ffmpeg -y -loglevel error -i traffic_density_analysis_raw.mp4 -vcodec libx264 -crf 23 traffic_density_analysis.mp4

if os.path.exists('traffic_density_analysis.mp4'):
    display(Video('traffic_density_analysis.mp4', embed=True, width=960))
else:
    print('Video conversion failed. Check ffmpeg output above.')

## 12. License Plate Detection & OCR

In [ ]:
import os
import logging
logging.disable(logging.WARNING)

from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang='en',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO


def read_plate_with_paddleocr(plate_crop: np.ndarray, ocr: 'PaddleOCR'):
    if plate_crop is None or plate_crop.size == 0:
        return '', 0.0

    h, w = plate_crop.shape[:2]
    if h < 32 or w < 64:
        scale      = max(32 / h, 64 / w)
        plate_crop = cv2.resize(
            plate_crop,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_CUBIC,
        )

    results = ocr.predict(plate_crop)

    if not results:
        return '', 0.0

    result = results[0]
    texts  = result.get('rec_texts',  []) if hasattr(result, 'get') else result['rec_texts']
    scores = result.get('rec_scores', []) if hasattr(result, 'get') else result['rec_scores']
    if not texts and hasattr(result, 'rec_texts'):
        texts  = result.rec_texts
        scores = result.rec_scores

    if not texts:
        return '', 0.0

    plate_text  = ' '.join(str(t) for t in texts).strip()
    avg_conf    = float(sum(scores) / len(scores)) if scores else 0.0
    return plate_text, avg_conf


def detect_and_read_plate(image_path: str, model: YOLO, ocr: 'PaddleOCR', conf: float = 0.5):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f'Image not found: {image_path}')

    yolo_results = model.predict(source=image_path, imgsz=640, conf=conf, verbose=False)
    boxes        = yolo_results[0].boxes
    names        = model.names

    plates = []

    if boxes is None or len(boxes) == 0:
        return plates

    for box in boxes:
        cls_id   = int(box.cls.item())
        cls_name = names.get(cls_id, '')
        if 'plate' not in cls_name.lower():
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        plate_crop      = image[y1:y2, x1:x2]

        plate_text, avg_conf = read_plate_with_paddleocr(plate_crop, ocr)
        plates.append((plate_text, avg_conf, (x1, y1, x2, y2)))

    return plates

if test_image_files:
    demo_path = os.path.join(test_images_path, test_image_files[0])
    detected_plates = detect_and_read_plate(demo_path, best_model, ocr)

    image_demo = cv2.imread(demo_path)
    if detected_plates:
        for plate_text, conf_score, (x1, y1, x2, y2) in detected_plates:
            print(f'Plate: "{plate_text}"  OCR confidence: {conf_score:.3f}')
            cv2.rectangle(image_demo, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(
                image_demo, plate_text,
                (x1, max(y1 - 10, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
            )
    else:
        print('No license plates detected in this image.')

    plt.figure(figsize=(14, 9))
    plt.imshow(cv2.cvtColor(image_demo, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('License Plate Detection + PaddleOCR 3.x', fontsize=16)
    plt.show()
else:
    print('No test images available.')

## 13. Contour-Based Plate Extraction

In [ ]:
def classical_plate_ocr(image_path: str, ocr: 'PaddleOCR'):
    image = cv2.imread(image_path)
    if image is None:
        print(f'Image not found: {image_path}')
        return

    gray  = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 30, 200)

    contours_raw, _ = cv2.findContours(
        edges.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )
    contours = sorted(contours_raw, key=cv2.contourArea, reverse=True)[:20]

    plate_contour = None
    for contour in contours:
        peri  = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
        if len(approx) == 4:
            plate_contour = approx
            break

    if plate_contour is None:
        print('License plate not detected via contours.')
        return

    x, y, w, h = cv2.boundingRect(plate_contour)
    plate_crop  = image[y:y+h, x:x+w]

    plate_text, _ = read_plate_with_paddleocr(plate_crop, ocr)
    print(f'License plate number: {plate_text}')

    cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 3)
    cv2.putText(
        image, plate_text, (max(x-50, 0), max(y-15, 0)),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Detected Plate: {plate_text}', fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Cropped Licence Plate', fontsize=13)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


if test_image_files:
    classical_plate_ocr(os.path.join(test_images_path, test_image_files[0]), ocr)

## 14. Export Weights for Backend Deployment

In [ ]:
export_path = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17,
)
print(f'ONNX model exported to: {export_path}')
import shutil
shutil.copy(best_model_path, '/content/best.pt')
print('best.pt copied to /content/best.pt — download from the Files panel.')
if export_path:
    shutil.copy(str(export_path), '/content/best.onnx')
    print('best.onnx copied to /content/best.onnx')



## Summary — What This Notebook Produces

| Artifact | Purpose |
|---|---|
| `best.pt` | Fine-tuned YOLOv8n weights — drop into `backend/models/vehicle.pt` |
| `best.onnx` | ONNX export for `onnxruntime` inference |
| `results.csv` | Training metrics (loss, mAP) per epoch |
| `confusion_matrix_normalized.png` | Class-level detection accuracy |

**Backend integration:**
- `backend/models/vehicle.pt` → loaded in `services/detector.py` via `YOLO('models/vehicle.pt')`
- `services/tracker.py` adds ByteTrack via `model.track(source, tracker='bytetrack.yaml')`
- `services/ocr.py` wraps `PaddleOCR.ocr()` as shown in Cell 20
- `services/violation.py` implements the rule engine on tracked detections
- `services/fine.py` generates JSON fine records sent to the database
